<a href="https://colab.research.google.com/github/Reina-creator/MSSP607-GitHub/blob/main/WeeklyModules/Week12/Week12Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import subprocess
from pathlib import Path

owner = "Reina-creator"
repository = "MSSP607-GitHub"
repo_url = f"https://github.com/{owner}/{repository}.git"
repo_path = Path("/content") / repository

if not repo_path.exists():
    subprocess.run(
        ["git", "clone", repo_url, str(repo_path)],
        check=True
    )
else:
    subprocess.run(
        ["git", "-C", str(repo_path), "pull", "--ff-only"],
        check=True
    )

os.chdir(repo_path)
print("Current directory:", os.getcwd())

Current directory: /content/MSSP607-GitHub


In [2]:
import pandas as pd
from scipy.stats import ttest_ind

data_file = repo_path / "data" / "Student Performance Data.csv"
students = pd.read_csv(data_file)

students[["gender", "test preparation course", "total score"]].head()

,gender,test preparation course,total score
0,female,none,218
1,female,completed,247
2,female,none,278
3,male,none,148
4,male,none,229


In [3]:
prep_col = "test preparation course"
score_col = "total score"

summary = (
    students.groupby(["gender", prep_col])[score_col]
    .agg(["count", "mean", "std"])
    .round(2)
)

summary

count    mean    std
gender test preparation course                      
female completed                  184  223.36  37.98
       none                       334  200.63  44.48
male   completed                  174  212.34  39.60
       none                       308  189.13  39.59

In [4]:
is_male = students["gender"] == "male"
completed_prep = students[prep_col] == "completed"

male_completed = is_male & completed_prep

group_1 = students.loc[male_completed, score_col]
group_2 = students.loc[~male_completed, score_col]

print(f"Male students who completed prep: n = {len(group_1)}, mean = {group_1.mean():.2f}")
print(f"All other students: n = {len(group_2)}, mean = {group_2.mean():.2f}")

Male students who completed prep: n = 174, mean = 212.34
All other students: n = 826, mean = 201.41


In [5]:
for gender in ["male", "female"]:
    same_gender = students["gender"] == gender

    completed_scores = students.loc[
        same_gender & completed_prep, score_col
    ]
    no_prep_scores = students.loc[
        same_gender & (students[prep_col] == "none"), score_col
    ]

    test = ttest_ind(completed_scores, no_prep_scores, equal_var=False)

    print(f"\n{gender.capitalize()} students")
    print(f"Completed prep mean: {completed_scores.mean():.2f}")
    print(f"No prep mean: {no_prep_scores.mean():.2f}")
    print(f"Mean difference: {completed_scores.mean() - no_prep_scores.mean():.2f}")
    print(f"p-value: {test.pvalue:.3e}")


Male students
Completed prep mean: 212.34
No prep mean: 189.13
Mean difference: 23.21
p-value: 1.726e-09

Female students
Completed prep mean: 223.36
No prep mean: 200.63
Mean difference: 22.73
p-value: 2.027e-09


Male students who completed the test preparation course had a higher mean total score than all other students (212.34 vs. 201.41). The difference was statistically significant in a Welch’s t-test (t = 3.257, p = .00127). In separate comparisons, students who completed the course scored about 23 points higher on average than students of the same gender who did not participate. These findings show an association between course completion and higher scores, but they do not establish that the course caused the difference.